# CGC ESI hardware probe (run before enabling the plugin)

This notebook identifies the real CGC module addresses and reads controller, HVPS-3kB, and HEAT-CTRL-2410 diagnostics directly through `COM-ESI-CTRL.dll`.

## Safety

- Run this notebook on the Windows PC connected to the ESI controller.
- Close ESIBD Explorer and the vendor ESI Control application first. The DLL is single-instance.
- The probe disables module operation before and after inventory. After identifying modules, it only writes safe zero targets (`0 V` and `0 degC`); it never activates a module or writes a nonzero target.
- It never activates a module and never changes an HV target or heater target.
- Existing outputs will be disabled and are not restored automatically.
- Keep the physical interlock available and independently verify that HV is off.
- The low-level vendor DLL has no cancellable timeout in this notebook. Power the controller on before probing; if a call blocks, restart the Jupyter kernel before opening ESIBD Explorer.

Run the cells in order. The final probe cell always closes the COM port, including after an exception.

In [ ]:
# Adjust only these values before running the probe.
COM_PORT = 16
BAUD_RATE = 230400

from pathlib import Path

REPO_ROOT = Path('C:/Users/lab_admin/ESIBD Explorer/plugins')
if not (REPO_ROOT / 'esi' / 'esi_plugin.py').is_file():
    raise FileNotFoundError(f'ESI plugin not found below {REPO_ROOT}')

DRIVER_FILE = REPO_ROOT / 'esi' / 'vendor' / 'runtime' / 'esi' / 'esi_base.py'
DLL_FILE = REPO_ROOT / 'esi' / 'vendor' / 'runtime' / 'esi' / 'vendor' / 'x64' / 'COM-ESI-CTRL.dll'
ERROR_CODES_FILE = REPO_ROOT / 'esi' / 'vendor' / 'runtime' / 'error_codes.json'

print(f'Repository: {REPO_ROOT}')
print(f'DLL:        {DLL_FILE}')
print(f'Port:       COM{COM_PORT} @ {BAUD_RATE} baud')

In [ ]:
import importlib.util
import json
from datetime import datetime

spec = importlib.util.spec_from_file_location('_esi_hardware_probe_base', DRIVER_FILE)
if spec is None or spec.loader is None:
    raise ImportError(f'Could not load ESI driver from {DRIVER_FILE}')
driver_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(driver_module)
ESIBase = driver_module.ESIBase

def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (tuple, list)):
        return [json_safe(item) for item in value]
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)

def status_result(device, action, result, *, required=True):
    if not isinstance(result, tuple):
        status, values = int(result), []
    else:
        status, values = int(result[0]), list(result[1:])
    record = {
        'status': status,
        'status_text': device.format_status(status),
        'values': json_safe(values),
    }
    if required and status != device.NO_ERR:
        raise RuntimeError(f'{action} failed: {device.format_status(status)}')
    return record

def optional_call(device, action, method, *args):
    try:
        return status_result(device, action, method(*args), required=False)
    except Exception as exc:
        return {'error': f'{type(exc).__name__}: {exc}'}

print('Low-level driver loaded. No COM port has been opened yet.')

## Run the safe probe

This cell opens the port, disables module operation, enables read-only inventory communication, inventories all addresses without assuming their mapping, reads diagnostics, saves a JSON report, disables module operation again, and closes the port.

In [ ]:
def run_probe():
    device = ESIBase(
        com=COM_PORT,
        idn='esi_hardware_probe',
        dll_path=DLL_FILE,
        error_codes_path=ERROR_CODES_FILE,
    )
    opened = False
    report = {
        'timestamp': datetime.now().astimezone().isoformat(),
        'com_port': COM_PORT,
        'requested_baud_rate': BAUD_RATE,
        'controller': {},
        'presence': {},
        'modules': {},
        'safety': {
            'module_operation_disabled_before_inventory': False,
            'module_activation_changed': False,
            'targets_changed': False,
        },
    }

    try:
        status_result(device, 'open_port', device.open_port(COM_PORT))
        opened = True

        baud = status_result(device, 'set_comspeed', device.set_comspeed(BAUD_RATE))
        report['actual_baud_rate'] = baud['values'][0]

        device_type = status_result(device, 'get_dev_type', device.get_dev_type())
        actual_device_type = int(device_type['values'][0])
        report['controller']['device_type'] = actual_device_type
        report['controller']['expected_device_type'] = device.DEVICE_TYPE
        if actual_device_type != device.DEVICE_TYPE:
            raise RuntimeError(
                f'Wrong controller: expected 0x{device.DEVICE_TYPE:04X}, '
                f'got 0x{actual_device_type:04X}'
            )

        # Close the global output gate, then reopen it only for module communication.
        status_result(device, 'set_enable(False)', device.set_enable(False))
        report['safety']['module_operation_disabled_before_inventory'] = True
        status_result(device, 'set_enable(True)', device.set_enable(True))
        enabled_state = status_result(device, 'get_enable', device.get_enable())
        report['safety']['inventory_communication_enabled'] = bool(
            enabled_state['values'][0]
        )

        status_result(device, 'update_module_presence', device.update_module_presence())
        presence_result = status_result(
            device, 'get_module_presence', device.get_module_presence()
        )
        valid, max_module, presence = presence_result['values']
        report['presence'] = {
            'valid': bool(valid),
            'max_module': int(max_module),
            'raw': presence,
            'base_state': int(presence[device.PRESENCE_BASE]),
        }
        if not valid:
            raise RuntimeError('Controller returned an invalid module-presence table.')

        controller_calls = {
            'product_id': device.get_product_id,
            'product_no': device.get_product_no,
            'firmware_version': device.get_fw_version,
            'firmware_date': device.get_fw_date,
            'hardware_type': device.get_hw_type,
            'hardware_version': device.get_hw_version,
            'main_state': device.get_main_state,
            'device_state': device.get_device_state,
            'voltage_state': device.get_voltage_state,
            'temperature_state': device.get_temperature_state,
            'fan_state': device.get_fan_state,
            'interlock_state': device.get_interlock_state,
            'interlock_enable': device.get_interlock_enable,
            'inputs': device.get_inputs,
            'housekeeping': device.get_housekeeping,
        }
        for name, method in controller_calls.items():
            report['controller'][name] = optional_call(device, name, method)

        for address in range(device.MODULE_NUM):
            presence_state = int(presence[address])
            module = {
                'address': address,
                'presence_state': presence_state,
                'presence_label': {
                    device.MODULE_NOT_FOUND: 'not found',
                    device.MODULE_PRESENT: 'present',
                    device.MODULE_INVALID: 'invalid',
                }.get(presence_state, 'unknown'),
            }
            report['modules'][str(address)] = module
            if presence_state != device.MODULE_PRESENT:
                continue

            identity_calls = {
                'device_type': device.get_module_dev_type,
                'product_id': device.get_module_product_id,
                'product_no': device.get_module_product_no,
                'hardware_type': device.get_module_hw_type,
                'hardware_version': device.get_module_hw_version,
                'firmware_version': device.get_module_fw_version,
                'state': device.get_module_state,
            }
            for name, method in identity_calls.items():
                module[name] = optional_call(device, f'{name}[{address}]', method, address)

            type_values = module['device_type'].get('values', [])
            module_type = int(type_values[0]) if type_values else None
            module['device_type_hex'] = (
                f'0x{module_type:04X}' if module_type is not None else None
            )

            if module_type == device.MODULE_HVPS_TYPE:
                module['role'] = 'HVPS-3kB'
                module['safe_zero_target'] = optional_call(
                    device, f'safe_zero_target[{address}]',
                    device.set_hv_supply_target_output_voltage, address, 0.0
                )
                module['fpga_version'] = optional_call(
                    device, f'fpga_version[{address}]',
                    device.get_hv_supply_fpga_version, address
                )
                module['target_voltage'] = optional_call(
                    device, f'target_voltage[{address}]',
                    device.get_hv_supply_target_output_voltage, address
                )
                module['pwm'] = optional_call(
                    device, f'pwm[{address}]',
                    device.get_hv_supply_params_pwm, address
                )
                module['measured_voltage'] = optional_call(
                    device, f'measured_voltage[{address}]',
                    device.get_hv_supply_output_voltage, address
                )
                module['measured_current'] = optional_call(
                    device, f'measured_current[{address}]',
                    device.get_hv_supply_output_current, address
                )
                module['housekeeping'] = optional_call(
                    device, f'hv_housekeeping[{address}]',
                    device.get_hv_supply_housekeeping, address
                )
            elif module_type == device.MODULE_HTCTRL_TYPE:
                module['role'] = 'HEAT-CTRL-2410'
                module['safe_zero_target'] = optional_call(
                    device, 'safe_heat_zero_target',
                    device.set_heat_ctrl_heater_temperature, 0.0
                )
                heat_calls = {
                    'hardware_limits': device.get_heat_ctrl_hw_limits,
                    'voltage_limit': device.get_heat_ctrl_voltage_limit,
                    'current_limit': device.get_heat_ctrl_current_limit,
                    'power_limit': device.get_heat_ctrl_power_limit,
                    'target_temperature': device.get_heat_ctrl_heater_temperature,
                    'output_voltage': device.get_heat_ctrl_output_voltage,
                    'heater_power': device.get_heat_ctrl_heater_power,
                    'monitoring': device.get_heat_ctrl_monitoring,
                    'interlock': device.get_heat_ctrl_ilock_state,
                    'housekeeping': device.get_heat_ctrl_housekeeping,
                }
                for name, method in heat_calls.items():
                    module[name] = optional_call(device, f'heat_{name}', method)
            else:
                module['role'] = 'unknown module type'

        heat_addresses = [
            int(address) for address, module in report['modules'].items()
            if module.get('role') == 'HEAT-CTRL-2410'
        ]
        hv_addresses = [
            int(address) for address, module in report['modules'].items()
            if module.get('role') == 'HVPS-3kB'
        ]
        report['recommended_mapping'] = {
            'ESI_HEAT': heat_addresses[0] if len(heat_addresses) == 1 else None,
            'ESI_HV1': hv_addresses[0] if len(hv_addresses) >= 1 else None,
            'ESI_HV2': hv_addresses[1] if len(hv_addresses) >= 2 else None,
        }

        report_path = (
            REPO_ROOT / 'esi' /
            f"esi_hardware_probe_{datetime.now():%Y%m%d_%H%M%S}.json"
        )
        report_path.write_text(
            json.dumps(json_safe(report), indent=2), encoding='utf-8'
        )

        print('\nDetected module map')
        print('-------------------')
        for address in range(device.MODULE_NUM):
            module = report['modules'][str(address)]
            print(
                f"address {address}: {module['presence_label']}; "
                f"type={module.get('device_type_hex')}; "
                f"role={module.get('role', 'n/a')}"
            )
        print(f"\nRecommended plugin mapping: {report['recommended_mapping']}")
        print(f'Report written to: {report_path}')
        return report, report_path
    finally:
        if opened:
            try:
                disable_status = device.set_enable(False)
                print(
                    'Module disable status: '
                    f'{device.format_status(disable_status)}'
                )
            except Exception as exc:
                print(f'WARNING: failed to disable modules: {exc}')
            try:
                close_status = device.close_port()
                print(f'COM port close status: {device.format_status(close_status)}')
            except Exception as exc:
                print(f'WARNING: failed to close COM port: {exc}')

report, report_path = run_probe()

## Interpretation

For the configuration inferred from earlier hardware tests, the expected mapping is:

```text
ESI_HEAT -> address 0, type 0xDB1C
ESI_HV1  -> address 1, type 0x0A0D
ESI_HV2  -> address 2, type 0x0A0D
address 3 -> not found
```

Do not enable the ESIBD plugin if the detected module types or mapping differ. Review the generated JSON report first.